# Stuttering Detection: Probabilistic Classification Analysis
**Course**: CS204T (Artificial Intelligence)  
**Team**: 18  
**Focus**: Naive Bayes vs. Normal Bayes (LDA)

---

## Step 1: Environment Setup

In [2]:
import os
import shutil
import numpy as np
from src.extractors import WavLMExtractor
from src.data import DataManager
from src.models import NaiveBayesModel, LDAModel

# Paths to our distributed dataset
AUDIO_DIR = "Stuttering Events in Podcasts Dataset/clips/stuttering-clips/clips"
CSV_PATHS = [
    "Stuttering Events in Podcasts Dataset/SEP-28k_labels.csv",
    "Stuttering Events in Podcasts Dataset/fluencybank_labels.csv"
]
FEATURE_DIR = "data/features"
fluent_dir = os.path.join(FEATURE_DIR, "fluent")
disfluent_dir = os.path.join(FEATURE_DIR, "disfluent")

## Step 2 (Optional): Operational Mode for Data Extraction
* `SKIP_EXTRACTION`: Uses features already on disk (Default).
* `FORCE_EXTRACT`: Analyzes raw audio for new files (Resumable).
* `CLEAN_START`: Wipes the database and re-extracts from zero.

In [ ]:
# Operational Flags
SKIP_EXTRACTION = True
FORCE_EXTRACT = False
CLEAN_START = False
NUM_CLIPS_TO_EXTRACT = 1000

if CLEAN_START:
    if os.path.exists(FEATURE_DIR):
        shutil.rmtree(FEATURE_DIR)
    print("[System] Clean start initiated. Wiped feature database.")

if not SKIP_EXTRACTION or CLEAN_START or FORCE_EXTRACT:
    extractor = WavLMExtractor("microsoft/wavlm-base")
    label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
    
    # Now using NATIVE Random Sampling logic for diversity
    extractor.extract_from_dir(
        AUDIO_DIR, 
        output_dir=FEATURE_DIR, 
        label_dict=label_dict, 
        limit=NUM_CLIPS_TO_EXTRACT, 
        random_sample=True
    )
else:
    print("[System] Skipping extraction. Using existing data on disk.")

## Step 3: Data Loading and Pipeline

In [3]:
label_dict = DataManager.generate_label_dict(CSV_PATHS, filter_quality=True)
manager = DataManager(None, None)

# Loading distributed .npy files
X, y = manager.load_from_folders(fluent_dir, disfluent_dir)
X_train, X_val, X_test, y_train, y_val, y_test = manager.get_splits(test_size=0.15, val_size=0.15)

# Balancing classes for accurate probability estimation
X_train_bal, y_train_bal = manager.balance_data(X_train, y_train, strategy="oversample")

# Standard Scaling (Pivotal for Gaussian Bayes)
X_train_final = manager.preprocess(X_train_bal, method="standard")
X_test_final = manager.preprocess(X_test, method="standard")

[DataManager] Quality Filter: Removed 3938 low-quality samples.


## Step 4: Model 1 - Naive Bayes

In [4]:
nb_model = NaiveBayesModel("Gaussian_Naive_Bayes")
nb_model.train(X_train_final, y_train_bal)
nb_model.evaluate(X_test_final, y_test)

[Model: Gaussian_Naive_Bayes] Initialized.
[Gaussian_Naive_Bayes] Training on 4440 samples...

--- Evaluation: Gaussian_Naive_Bayes ---
Accuracy: 0.6370
Precision: 0.4943
Recall: 0.6566
F1: 0.5640

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      298             178            
True: Stutter(1)     91              174            


{'accuracy': 0.6369770580296896,
 'precision': 0.4943181818181818,
 'recall': 0.6566037735849056,
 'f1': 0.5640194489465153,
 'confusion_matrix': array([[298, 178],
        [ 91, 174]])}

## Step 5: Model 2 - Normal Bayes (LDA)

In [6]:
lda_model = LDAModel("Normal_Bayes_LDA")
lda_model.train(X_train_final, y_train_bal)
lda_model.evaluate(X_test_final, y_test)

[Model: Normal_Bayes_LDA] Initialized.
[Normal_Bayes_LDA] Training Bayesian Probability Map...

--- Evaluation: Normal_Bayes_LDA ---
Accuracy: 0.6383
Precision: 0.4958
Recall: 0.6679
F1: 0.5691

Confusion Matrix (Binary):
               Predicted: Fluent(0)  Predicted: Stutter(1)
True: Fluent(0)      296             180            
True: Stutter(1)     88              177            


{'accuracy': 0.6383265856950068,
 'precision': 0.4957983193277311,
 'recall': 0.6679245283018868,
 'f1': 0.5691318327974276,
 'confusion_matrix': array([[296, 180],
        [ 88, 177]])}